# <center> <img src="../img/ITESOLogo.png" alt="ITESO" width="480" height="130"> </center>
# <center> **Departamento de Electrónica, Sistemas e Informática** </center>
---
## <center> **Big Data** </center>
---
### <center> **Spring 2026** </center>
---
### <center> **Examples on Structured Streaming (sockets)** </center>
---
**Profesor**: Pablo Camarillo Ramirez

# Create SparkSession

In [1]:
import findspark
findspark.init()

from spark_utils import SparkUtils

su = SparkUtils("Examples on Structured Streaming",
                   master_url="spark://spark-master:7077")

su.spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/26 01:40:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


# Create a data stream from a local socket

### Install netcat utility

In [2]:
!apt-get update
!apt-get install -y netcat

Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Reading package lists... Done
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
netcat is already the newest version (1.218-4ubuntu1).
0 upgraded, 0 newly installed, 0 to remove and 36 not upgraded.


# Correr en la terminal del nootbook
nc -lk 9999

### Connect Spark to the socket

In [3]:
import pyspark.sql.functions as F

# Create the remote connection
lines = su.spark.readStream \
            .format("socket") \
            .option("host", "localhost") \
            .option("port", 9999) \
            .load()

# Perform some transformations to the input data (word counter)
words = lines.select(F.explode(F.split(lines.value, " ")).alias("word"))
word_count = words.groupBy("word").count()

# Send transformed data to the Sink
query = word_count.writeStream \
            .outputMode("complete") \
            .format("console") \
            .start()
query.awaitTermination(300)


26/03/26 01:41:25 WARN TextSocketSourceProvider: The socket source should not be used for production applications! It does not support recovery.
26/03/26 01:41:26 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-1e5ba5cf-3787-4800-a105-e8ffbce9c096. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/03/26 01:41:26 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
                                                                                

-------------------------------------------
Batch: 0
-------------------------------------------
+----+-----+
|word|count|
+----+-----+
+----+-----+



-------------------------------------------
Batch: 1
-------------------------------------------
+----+-----+
|word|count|
+----+-----+
|    |    1|
+----+-----+



-------------------------------------------
Batch: 2
-------------------------------------------
+----+-----+
|word|count|
+----+-----+
|  hi|    1|
|    |    1|
+----+-----+



-------------------------------------------
Batch: 3
-------------------------------------------
+--------------------+-----+
|                word|count|
+--------------------+-----+
|                  hi|    1|
|AAAAAAAAAAAAAAAAA...|    1|
|                    |    1|
+--------------------+-----+



...

-------------------------------------------
Batch: 5
-------------------------------------------
+--------------------+-----+
|                word|count|
+--------------------+-----+
|                  hi|    1|
|AAAAAAAAAAAAAAAAA...|    1|
|              Pepito|    1|
|                 ALO|    1|
|                    |    1|
+--------------------+-----+

-------------------------------------------
Batch: 6
-------------------------------------------
+--------------------+-----+
|                word|count|
+--------------------+-----+
|AAAAAAAAAAAAAAAAA...|    1|
|              Pepito|    1|
|                  hi|    1|
|                 ALO|    1|
|                    |    1|
|               clavó|    1|
+--------------------+-----+



-------------------------------------------
Batch: 7
-------------------------------------------
+--------------------+-----+
|                word|count|
+--------------------+-----+
|AAAAAAAAAAAAAAAAA...|    1|
|              Pepito|    1|
|                  hi|    1|
|                 ALO|    1|
|                  un|    1|
|                    |    1|
|               clavó|    1|
|             clavito|    1|
+--------------------+-----+



-------------------------------------------
Batch: 8
-------------------------------------------
+--------------------+-----+
|                word|count|
+--------------------+-----+
|AAAAAAAAAAAAAAAAA...|    1|
|              Pepito|    1|
|                  hi|    1|
|                 ALO|    1|
|                  un|    2|
|                    |    1|
|               clavó|    1|
|             clavito|    1|
+--------------------+-----+



-------------------------------------------
Batch: 9
-------------------------------------------
+--------------------+-----+
|                word|count|
+--------------------+-----+
|AAAAAAAAAAAAAAAAA...|    1|
|              Pepito|    1|
|                  hi|    1|
|asdfasndfasdf,asm...|    1|
|                 ALO|    1|
|                  un|    2|
|                    |    1|
|               clavó|    1|
|             clavito|    1|
+--------------------+-----+

-------------------------------------------
Batch: 10
-------------------------------------------
+--------------------+-----+
|                word|count|
+--------------------+-----+
|AAAAAAAAAAAAAAAAA...|    1|
|              Pepito|    1|
| msdnfabsdhjfasdavdf|    1|
|                  hi|    1|
|                 ALO|    1|
|asdfasndfasdf,asm...|    1|
|                  un|    2|
|                    |    1|
|               clavó|    1|
|             clavito|    1|
+--------------------+-----+



False

In [4]:
su.spark.stop()